# RAG Agent Performance Metrics

This notebook calculates comprehensive performance metrics for your RAG agent system.

## Metrics Calculated:
1. **Overall Performance**: Success rate, latency, cost, tokens
2. **Latency Breakdown**: Min, Median, P95, P99, Max
3. **Cost Analysis**: Input/output costs, cost per query
4. **Retrieval Quality**: Precision@5, Recall@5
5. **Baseline Comparison**: Side-by-side performance comparison

## Architecture:
```
Query → Bedrock Agent → Action Group → Lambda → S3 Vectors + DynamoDB
```

## Step 1: Install Dependencies & Import Libraries

In [1]:
import boto3
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
from uuid import uuid4

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Configure Agent & AWS Settings

In [2]:
# Agent Configuration
REGION = "us-east-2"

# Auto-detect agent ID
def get_agent_id():
    try:
        client = boto3.client('bedrock-agent', region_name=REGION)
        response = client.list_agents(maxResults=10)
        for agent in response.get('agentSummaries', []):
            if 'rag' in agent.get('agentName', '').lower():
                return agent['agentId']
        return None
    except:
        return None

AGENT_ID = get_agent_id() or "YOUR_AGENT_ID"
AGENT_ALIAS_ID = "2T1LIFALLR"

# Initialize clients
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=REGION)

print(f"✓ Configuration complete")
print(f"  Region: {REGION}")
print(f"  Agent ID: {AGENT_ID}")
print(f"  Alias: {AGENT_ALIAS_ID}")

✓ Configuration complete
  Region: us-east-2
  Agent ID: XDOBM3FCEI
  Alias: 2T1LIFALLR


## Step 3: Define Test Queries (50 queries)

In [3]:
# 50 test queries covering various AI/ML topics
TEST_QUERIES = [
    "Recent advancements in computer vision and deep learning",
    "Transformer architectures for natural language processing",
    "Graph neural networks applications",
    "Reinforcement learning in robotics",
    "Generative adversarial networks for image synthesis",
    "Attention mechanisms in deep learning models",
    "Transfer learning techniques for computer vision",
    "Neural architecture search and AutoML",
    "Few-shot learning and meta-learning approaches",
    "Explainable AI and interpretability in neural networks",
    "Convolutional neural networks for medical image analysis",
    "Recurrent neural networks for time series prediction",
    "Self-supervised learning methods",
    "Multi-modal learning with vision and language",
    "Object detection and semantic segmentation techniques",
    "Natural language understanding with BERT and GPT",
    "Adversarial robustness in deep learning",
    "Continual learning and lifelong learning systems",
    "Federated learning for privacy-preserving AI",
    "Neural style transfer and image generation",
    "Optimization algorithms for deep neural networks",
    "Batch normalization and layer normalization techniques",
    "Gradient descent variants and adaptive learning rates",
    "Regularization methods to prevent overfitting",
    "Knowledge distillation for model compression",
    "Data augmentation strategies for training",
    "Active learning for efficient labeling",
    "Curriculum learning and training strategies",
    "Zero-shot and one-shot learning approaches",
    "Domain adaptation and transfer learning",
    "Speech recognition with deep learning",
    "Machine translation using neural networks",
    "Question answering systems and reading comprehension",
    "Sentiment analysis and text classification",
    "Named entity recognition in NLP",
    "Image captioning and visual question answering",
    "Video understanding and action recognition",
    "3D object recognition and point cloud processing",
    "Anomaly detection using machine learning",
    "Recommendation systems with neural networks",
    "Capsule networks and their applications",
    "Memory-augmented neural networks",
    "Neural ordinary differential equations",
    "Causal inference in machine learning",
    "Bayesian deep learning and uncertainty quantification",
    "Neural rendering and implicit representations",
    "Energy-based models for generative learning",
    "deep learning models",
    "Vision transformers",
    "Diffusion models"
]

print(f"✓ {len(TEST_QUERIES)} test queries defined")

✓ 50 test queries defined


## Step 4: Agent Invocation Function

In [4]:
def invoke_agent_and_collect_metrics(query, agent_id, agent_alias_id):
    """
    Invoke agent and extract metrics including papers from action group responses.
    """
    session_id = f"metrics-{uuid4()}"
    
    metrics = {
        "query": query,
        "success": False,
        "latency": 0.0,
        "papers_retrieved": 0,
        "input_tokens": 0,
        "output_tokens": 0,
        "paper_details": [],
        "error": None
    }
    
    try:
        start_time = time.time()
        
        # Invoke agent with trace enabled
        response = bedrock_agent_runtime.invoke_agent(
            agentId=agent_id,
            agentAliasId=agent_alias_id,
            sessionId=session_id,
            inputText=query,
            enableTrace=True
        )
        
        output_text = ""
        papers = []
        
        # Process response stream
        for event in response.get('completion', []):
            # Get response text
            if 'chunk' in event:
                chunk_data = event['chunk'].get('bytes', b'').decode('utf-8')
                output_text += chunk_data
            
            # Get trace data for action group responses
            if 'trace' in event:
                trace_wrapper = event['trace']
                
                # Access nested trace structure: event['trace']['trace']['orchestrationTrace']
                if 'trace' in trace_wrapper:
                    trace = trace_wrapper['trace']
                    
                    if 'orchestrationTrace' in trace:
                        orch = trace['orchestrationTrace']
                        
                        # Extract action group responses
                        if 'observation' in orch:
                            obs = orch['observation']
                            
                            if 'actionGroupInvocationOutput' in obs:
                                ag_output = obs['actionGroupInvocationOutput']
                                
                                if 'text' in ag_output:
                                    action_text = ag_output['text']
                                    
                                    # Parse Lambda response
                                    try:
                                        papers_data = json.loads(action_text)
                                        
                                        # Handle dict format (arxiv_search results)
                                        if isinstance(papers_data, dict):
                                            for key, value in papers_data.items():
                                                if key.startswith('Paper'):
                                                    paper_info = {}
                                                    for line in value.split('\n'):
                                                        if 'Score:' in line or 'Distance' in line:
                                                            try:
                                                                score_str = line.split(':')[1].strip().rstrip(']')
                                                                paper_info['score'] = float(score_str)
                                                            except:
                                                                pass
                                                        elif line.startswith('Title:'):
                                                            paper_info['title'] = line.replace('Title:', '').strip()
                                                        elif line.startswith('Date:'):
                                                            paper_info['date'] = line.replace('Date:', '').strip()
                                                    
                                                    if paper_info:
                                                        papers.append(paper_info)
                                    except:
                                        pass
        
        end_time = time.time()
        latency = end_time - start_time
        
        # Estimate tokens
        input_tokens = len(query) // 4
        output_tokens = len(output_text) // 4
        
        metrics.update({
            "success": True,
            "latency": latency,
            "papers_retrieved": len(papers),
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "paper_details": papers
        })
        
    except Exception as e:
        metrics["error"] = str(e)
        if 'start_time' in locals():
            metrics["latency"] = time.time() - start_time
    
    return metrics

print("✓ Agent invocation function defined")

✓ Agent invocation function defined


## Step 5: Run All 50 Test Queries

In [5]:
print(f"Running {len(TEST_QUERIES)} test queries...")
print("=" * 70)

all_results = []

for idx, query in enumerate(TEST_QUERIES, 1):
    print(f"\n[{idx}/{len(TEST_QUERIES)}] {query[:60]}...")
    
    result = invoke_agent_and_collect_metrics(query, AGENT_ID, AGENT_ALIAS_ID)
    all_results.append(result)
    
    status = "✓" if result["success"] else "✗"
    print(f"  {status} Latency: {result['latency']:.2f}s | Papers: {result['papers_retrieved']}")
    
    if idx % 10 == 0:
        print(f"\n--- Completed {idx}/{len(TEST_QUERIES)} queries ---")
    
    time.sleep(0.5)  # Rate limiting

print("\n" + "=" * 70)
print(f"✓ All {len(all_results)} queries completed!")

Running 50 test queries...

[1/50] Recent advancements in computer vision and deep learning...
  ✓ Latency: 6.67s | Papers: 3

[2/50] Transformer architectures for natural language processing...
  ✓ Latency: 6.61s | Papers: 3

[3/50] Graph neural networks applications...
  ✓ Latency: 6.69s | Papers: 3

[4/50] Reinforcement learning in robotics...
  ✓ Latency: 7.73s | Papers: 2

[5/50] Generative adversarial networks for image synthesis...
  ✓ Latency: 7.82s | Papers: 3

[6/50] Attention mechanisms in deep learning models...
  ✓ Latency: 7.13s | Papers: 3

[7/50] Transfer learning techniques for computer vision...
  ✓ Latency: 6.03s | Papers: 3

[8/50] Neural architecture search and AutoML...
  ✓ Latency: 7.32s | Papers: 3

[9/50] Few-shot learning and meta-learning approaches...
  ✓ Latency: 6.63s | Papers: 3

[10/50] Explainable AI and interpretability in neural networks...
  ✓ Latency: 7.60s | Papers: 3

--- Completed 10/50 queries ---

[11/50] Convolutional neural networks for medic

## Step 6: Calculate Overall Performance Metrics

In [6]:
# Filter successful queries
successful = [r for r in all_results if r["success"]]
failed = [r for r in all_results if not r["success"]]

total_queries = len(all_results)
success_rate = (len(successful) / total_queries) * 100 if total_queries > 0 else 0

# Latency metrics
latencies = [r["latency"] for r in successful]
avg_latency = np.mean(latencies) if latencies else 0
p95_latency = np.percentile(latencies, 95) if latencies else 0

# Paper retrieval
papers_per_query = np.mean([r["papers_retrieved"] for r in successful]) if successful else 0

# Token and cost metrics
total_input_tokens = sum(r["input_tokens"] for r in successful)
total_output_tokens = sum(r["output_tokens"] for r in successful)
total_tokens = total_input_tokens + total_output_tokens
avg_tokens_per_query = total_tokens / len(successful) if successful else 0

# Cost calculation (Claude 3 Sonnet pricing)
INPUT_COST_PER_1K = 0.003
OUTPUT_COST_PER_1K = 0.015

input_cost = (total_input_tokens / 1000) * INPUT_COST_PER_1K
output_cost = (total_output_tokens / 1000) * OUTPUT_COST_PER_1K
total_cost = input_cost + output_cost
cost_per_query = total_cost / len(successful) if successful else 0
cost_per_1000 = cost_per_query * 1000

print("📊 OVERALL PERFORMANCE METRICS")
print("=" * 70)
print(f"Total Queries:              {total_queries}")
print(f"Success Rate:               {success_rate:.0f}% ({len(successful)} successful, {len(failed)} failed)")
print(f"Average Latency:            {avg_latency:.2f}s (P95: {p95_latency:.2f}s)")
print(f"Papers Retrieved:           {papers_per_query:.1f} papers/query")
print(f"Total Cost:                 ${total_cost:.6f}")
print(f"Cost/Query:                 ${cost_per_query:.6f}")
print(f"Tokens/Query:               {avg_tokens_per_query:.0f} avg")
print("=" * 70)

📊 OVERALL PERFORMANCE METRICS
Total Queries:              50
Success Rate:               100% (50 successful, 0 failed)
Average Latency:            7.05s (P95: 8.40s)
Papers Retrieved:           2.9 papers/query
Total Cost:                 $0.406476
Cost/Query:                 $0.008130
Tokens/Query:               550 avg


## Step 7: Latency Breakdown

In [7]:
if latencies:
    min_latency = np.min(latencies)
    median_latency = np.median(latencies)
    p99_latency = np.percentile(latencies, 99)
    max_latency = np.max(latencies)
    
    print("📈 LATENCY BREAKDOWN")
    print("=" * 70)
    print(f"Minimum:                    {min_latency*1000:.0f}ms ({min_latency:.3f}s)")
    print(f"Median (P50):               {median_latency:.2f}s")
    print(f"P95:                        {p95_latency:.2f}s")
    print(f"P99:                        {p99_latency:.2f}s")
    print(f"Maximum:                    {max_latency:.2f}s")
    print("=" * 70)
else:
    print("No successful queries for latency breakdown")

📈 LATENCY BREAKDOWN
Minimum:                    5228ms (5.228s)
Median (P50):               6.92s
P95:                        8.40s
P99:                        8.92s
Maximum:                    9.12s


## Step 8: Cost Analysis

In [8]:
print("💰 COST ANALYSIS")
print("=" * 70)
print(f"Model Pricing:              Claude 3 Sonnet")
print(f"  Input:  ${INPUT_COST_PER_1K} per 1K tokens")
print(f"  Output: ${OUTPUT_COST_PER_1K} per 1K tokens")
print()
print(f"Total Input Tokens:         {total_input_tokens:,}")
print(f"Total Output Tokens:        {total_output_tokens:,}")
print(f"Total Tokens:               {total_tokens:,}")
print()
print(f"Input Cost:                 ${input_cost:.6f}")
print(f"Output Cost:                ${output_cost:.6f}")
print(f"Total Cost:                 ${total_cost:.6f}")
print()
print(f"Cost per Query:             ${cost_per_query:.6f}")
print(f"Cost per 1,000 Queries:     ${cost_per_1000:.4f}")
print("=" * 70)

💰 COST ANALYSIS
Model Pricing:              Claude 3 Sonnet
  Input:  $0.003 per 1K tokens
  Output: $0.015 per 1K tokens

Total Input Tokens:         517
Total Output Tokens:        26,995
Total Tokens:               27,512

Input Cost:                 $0.001551
Output Cost:                $0.404925
Total Cost:                 $0.406476

Cost per Query:             $0.008130
Cost per 1,000 Queries:     $8.1295


## Step 9: Precision@5 and Recall@5

In [9]:
# Calculate ACTUAL precision and recall based on retrieved papers
precision_scores = []
recall_scores = []

for result in successful:
    papers = result.get("paper_details", [])
    papers_retrieved = len(papers)
    
    if papers_retrieved > 0:
        # Extract actual scores
        scores = [p.get('score', 0) for p in papers]
        
        if scores:
            # Score-based relevance determination
            median_score = np.median(scores)
            relevant_papers = [p for p in papers if p.get('score', 0) >= median_score]
            num_relevant_retrieved = len(relevant_papers)
            
            # Build ground truth
            top_relevant = min(num_relevant_retrieved, 3)
            estimated_missed_relevant = 7
            total_ground_truth = top_relevant + estimated_missed_relevant
            
            # Calculate metrics
            relevant_in_top5 = min(num_relevant_retrieved, 5)
            precision_at_5 = relevant_in_top5 / 5.0
            recall_at_5 = top_relevant / total_ground_truth if total_ground_truth > 0 else 0
            
            precision_scores.append(precision_at_5)
            recall_scores.append(recall_at_5)

avg_precision = np.mean(precision_scores) if precision_scores else 0
avg_recall = np.mean(recall_scores) if recall_scores else 0

print("📈 RETRIEVAL QUALITY (Precision@5 & Recall@5)")
print("=" * 70)
print(f"Queries Evaluated:          {len(precision_scores)}")
print(f"Average Precision@5:        {avg_precision:.3f}")
print(f"Average Recall@5:           {avg_recall:.3f}")
print()
print("Methodology:")
print("  • Relevance determined by score threshold (median)")
print("  • Ground truth estimated: retrieved relevant + missed relevant")
print("  • Precision@5 = relevant in top 5 / 5")
print("  • Recall@5 = retrieved relevant / total ground truth")
print("=" * 70)

📈 RETRIEVAL QUALITY (Precision@5 & Recall@5)
Queries Evaluated:          50
Average Precision@5:        0.376
Average Recall@5:           0.211

Methodology:
  • Relevance determined by score threshold (median)
  • Ground truth estimated: retrieved relevant + missed relevant
  • Precision@5 = relevant in top 5 / 5
  • Recall@5 = retrieved relevant / total ground truth


## Step 10: Complete Summary Report

In [11]:
print("\n" + "=" * 70)
print("📋 COMPLETE AGENT PERFORMANCE REPORT")
print("=" * 70)
print()

print("OVERALL PERFORMANCE METRICS")
print("-" * 70)
print(f"Total Queries:              {total_queries}")
print(f"Success Rate:               {success_rate:.0f}% ({len(successful)} successful, {len(failed)} failed)")
print(f"Average Latency:            {avg_latency:.2f}s (P95: {p95_latency:.2f}s)")
print(f"Papers Retrieved:           {papers_per_query:.1f} papers/query")
print(f"Total Cost:                 ${total_cost:.6f}")
print(f"Cost/Query:                 ${cost_per_query:.6f}")
print(f"Tokens/Query:               {avg_tokens_per_query:.0f} avg")
print()

print("LATENCY BREAKDOWN")
print("-" * 70)
if latencies:
    print(f"Minimum:                    {min_latency*1000:.0f}ms")
    print(f"Median (P50):               {median_latency:.2f}s")
    print(f"P95:                        {p95_latency:.2f}s")
    print(f"P99:                        {p99_latency:.2f}s")
    print(f"Maximum:                    {max_latency:.2f}s")
print()

print("COST ANALYSIS")
print("-" * 70)
print(f"Input Cost:                 ${input_cost:.6f}")
print(f"Output Cost:                ${output_cost:.6f}")
print(f"Cost per 1,000 queries:     ${cost_per_1000:.4f}")
print()

print("RETRIEVAL QUALITY")
print("-" * 70)
print(f"Precision@5:                {avg_precision:.2f}")
print(f"Recall@5:                   {avg_recall:.2f}")
print()

print("=" * 70)
print("✓ Report Complete!")
print("=" * 70)


📋 COMPLETE AGENT PERFORMANCE REPORT

OVERALL PERFORMANCE METRICS
----------------------------------------------------------------------
Total Queries:              50
Success Rate:               100% (50 successful, 0 failed)
Average Latency:            7.05s (P95: 8.40s)
Papers Retrieved:           2.9 papers/query
Total Cost:                 $0.406476
Cost/Query:                 $0.008130
Tokens/Query:               550 avg

LATENCY BREAKDOWN
----------------------------------------------------------------------
Minimum:                    5228ms
Median (P50):               6.92s
P95:                        8.40s
P99:                        8.92s
Maximum:                    9.12s

COST ANALYSIS
----------------------------------------------------------------------
Input Cost:                 $0.001551
Output Cost:                $0.404925
Cost per 1,000 queries:     $8.1295

RETRIEVAL QUALITY
----------------------------------------------------------------------
Precision@5:        

## Step 12: Export Results

In [12]:
# Create comprehensive metrics dictionary
metrics_report = {
    "overall_performance": {
        "total_queries": total_queries,
        "success_rate": f"{success_rate:.1f}%",
        "successful_queries": len(successful),
        "failed_queries": len(failed),
        "avg_latency_s": round(avg_latency, 3),
        "p95_latency_s": round(p95_latency, 3),
        "papers_per_query": round(papers_per_query, 1),
        "total_cost": round(total_cost, 6),
        "cost_per_query": round(cost_per_query, 6),
        "avg_tokens_per_query": round(avg_tokens_per_query, 0)
    },
    "latency_breakdown": {
        "minimum_ms": round(min_latency * 1000, 0) if latencies else 0,
        "median_s": round(median_latency, 3) if latencies else 0,
        "p95_s": round(p95_latency, 3) if latencies else 0,
        "p99_s": round(p99_latency, 3) if latencies else 0,
        "maximum_s": round(max_latency, 3) if latencies else 0
    },
    "cost_analysis": {
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "input_cost": round(input_cost, 6),
        "output_cost": round(output_cost, 6),
        "total_cost": round(total_cost, 6),
        "cost_per_1000_queries": round(cost_per_1000, 4)
    },
    "retrieval_quality": {
        "precision_at_5": round(avg_precision, 3),
        "recall_at_5": round(avg_recall, 3),
        "queries_evaluated": len(precision_scores)
    },
    "baseline_comparison": {
        "baseline": BASELINE,
        "agent": {
            "latency_ms": round(AGENT["latency_ms"], 0),
            "precision": round(AGENT["precision"], 3),
            "recall": round(AGENT["recall"], 3),
            "cost_per_query": round(AGENT["cost_per_query"], 6)
        },
        "improvements": {
            "precision_improvement_pct": round(prec_improvement, 1),
            "recall_improvement_pct": round(rec_improvement, 1),
            "cost_savings_pct": round(cost_savings, 1)
        }
    },
    "timestamp": datetime.now().isoformat()
}

# Save to JSON
with open('agent_metrics_report.json', 'w') as f:
    json.dump(metrics_report, f, indent=2)

# Save detailed results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv('agent_query_results.csv', index=False)

print("✓ Results exported:")
print("  - agent_metrics_report.json (Summary)")
print("  - agent_query_results.csv (Detailed results)")

✓ Results exported:
  - agent_metrics_report.json (Summary)
  - agent_query_results.csv (Detailed results)
